In [ ]:
import numpy as np
import sleap
from sleap.info.write_tracking_h5 import get_occupancy_and_points_matrices
import os

# Print some info:
sleap.versions()
sleap.system_summary()

In [ ]:
def split_video_and_h5(sleap_filename):
    # Load labels and remove empty frames
    labels = sleap.load_file(sleap_filename)
    labels.remove_empty_frames()

    # Find start and end frames of bouts where the mouse is in frame
    # Get occupancy matrix
    matrices = get_occupancy_and_points_matrices(labels,all_frames=True)
    occupancy_matrix = np.squeeze(matrices[0])
    # Find the frames where occupancy_matrix is 1 (i.e. mouse is present)
    frames = np.where(occupancy_matrix == 1)[0]
    # Split the frames array to get subarrays of consecutive frames
    split_frames = np.split(frames, np.where(np.diff(frames) != 1)[0] + 1)
    # Filter the split_frames subarrays to remove those with less than 50 frames (arbitrary number, can be changed)
    filtered_split_frames = [subarray for subarray in split_frames if len(subarray) > 100]
    # Only keep the start and end frame of each subarray
    start_end_frame_pairs = [(subarr[0], subarr[-1]) for subarr in filtered_split_frames]

    # Split video and h5 files
    video = labels.video.backend.filename
    path_parts = list(os.path.split(video))
    path_parts.insert(-1, "split_data")
    new_path = os.path.join(*path_parts)

    for i, pair in enumerate(start_end_frame_pairs):
        print("Saving trimmed files number", i+1, "of", len(start_end_frame_pairs), "...")
            
        # Create a sub video from the original video using the start and end frames
        sub_video = new_path[0:-4] + "_trimmed" + f"_{i+1}" + new_path[-4:]
        start_frame = pair[0]
        end_frame = pair[1]
        %system ffmpeg -i {video} -vf "trim=start_frame={start_frame}:end_frame={end_frame},setpts=PTS-STARTPTS" -q:v 0 {sub_video}

        # Create sub sleap and h5 files from the original sleap file using the start and end frames
        inds = np.arange(start_frame, end_frame)
        sub_labels = labels.get((labels.video, inds))
        # Update the frame_idx of each LabeledFrame in the sub_labels
        sub_labels = reindex_frames(sub_labels)
        # Tranform the sub_labels into a Labels object
        sub_labels = sleap.Labels(sub_labels)
        # Update the video filename
        sub_labels.video.backend.filename = sub_video
        # Optionally save the sub sleap file (not necessary for MoSeq, can be commented out)
        # sub_sleap_file = new_path[0:-4] + "_trimmed" + f"_{i+1}" + ".slp"
        # sleap.Labels.save_file(sub_labels, sub_sleap_file)
        # Save the corresponding sub h5 file
        sub_h5_file = new_path[0:-4] + "_trimmed" + f"_{i+1}" + ".h5"
        sub_labels.export(sub_h5_file)

def reindex_frames(labels_list):
    for j in range(len(labels_list)):
        labels_list[j].frame_idx = j
    return labels_list

In [ ]:
sleap_filenames = ["predictions_for_moseq/CameraNestNorthTop_2023-06-28T15-00-00.slp",
                   "predictions_for_moseq/CameraNestNorthTop_2023-06-28T18-00-00.slp",
                   "predictions_for_moseq/CameraNestNorthTop_2023-07-03T18-00-00.slp",
                   "predictions_for_moseq/CameraNestNorthTop_2023-07-03T19-00-00.slp"]
for i, sleap_filename in enumerate(sleap_filenames):
    print("PROCESSING VIDEO", i+1, "OF", len(sleap_filenames))
    split_video_and_h5(sleap_filename)
print("Done")